In [1]:
import os, subprocess, sys

# Path for Debian/Ubuntu openjdk-17:
JAVA_HOME = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = JAVA_HOME + "/bin:" + os.environ["PATH"]

# quick sanity:
print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("java -version ->\n", subprocess.run(["java","-version"], capture_output=True, text=True).stderr)
import  sedona, pyspark
print("PySedona:", sedona.version)           # should be 1.8.0
print("Spark:", pyspark.__version__)             # should be 4.0.x

JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64
java -version ->
 openjdk version "21.0.9" 2025-10-21
OpenJDK Runtime Environment (build 21.0.9+10-Debian-1deb13u1)
OpenJDK 64-Bit Server VM (build 21.0.9+10-Debian-1deb13u1, mixed mode, sharing)

PySedona: 1.8.0
Spark: 4.0.1


In [3]:
from sedona.spark import *
from sedona.spark.register.geo_registrator import SedonaRegistrator
import os
import sys

# --- Configuration ---

# Choose ONE master setting:
# Note: For testing, "local[*]" is often easier if you are not in a controlled cluster environment.
MASTER = "spark://spark-master:7077" # cluster mode
# MASTER = "local[*]"                 # local fallback

# Using the single combined package and the Spark 4.0/Scala 2.13 naming convention as requested,
# but ensuring the geotools wrapper uses a stable, resolvable version.
SEDONA_PACKAGES = (
    "org.apache.sedona:sedona-spark-4.0_2.13:1.8.0,"
    "org.datasyslab:geotools-wrapper:1.8.0-28.2" # Reverted to the stable geotools version (1.8.0-28.2)
)

# --- Initialization ---

print(f"Initializing Spark Session with Master: {MASTER}")
print(f"Using Packages: {SEDONA_PACKAGES}")

# CRITICAL FIX: The result of .getOrCreate() is the initialized SedonaContext.
try:
    sedona = (
        SedonaContext.builder()
        .appName("GeoJoinBenchmark")
        .master(MASTER)
        .config(
            "spark.jars.packages",
            SEDONA_PACKAGES,
        )
        # The geotools-wrapper dependency often requires the Unidata repository.
        .config(
            "spark.jars.repositories",
            "https://artifacts.unidata.ucar.edu/repository/unidata-all",
        )
        # Recommended: Add Kryo serializer configs for optimized spatial data handling
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator")
        # Ensure the session starts with the new configs
        .getOrCreate()
    )

    # Explicitly register Sedona functions in the SQL context to resolve UNRESOLVED_ROUTINE errors
    SedonaRegistrator.registerAll(sedona)

    print("SedonaContext successfully created.")

    # --- Test Query ---
    # The SedonaContext object 'sedona' now inherits all SparkSession methods
    # and has the Sedona SQL functions registered.
    print("\nRunning Test Query:")
    sedona.sql("SELECT ST_AsText(ST_Point(0,0)) AS pt").show(truncate=False)

except Exception as e:
    print(f"\n--- ERROR DURING INITIALIZATION ---")
    print(f"An error occurred: {e}")
    print("If the error is related to 'Unresolved Dependencies' or 'ClassNotFound', it confirms a dependency or environment mismatch. Please ensure you have restarted your kernel/session.")

# Clean up when done (optional, but good practice in standalone scripts)
# sedona.stop()

Initializing Spark Session with Master: spark://spark-master:7077
Using Packages: org.apache.sedona:sedona-spark-4.0_2.13:1.8.0,org.datasyslab:geotools-wrapper:1.8.0-28.2


25/11/13 05:40:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
/tmp/ipykernel_2182/1329272452.py:48: DeprecationWarning: Call to deprecated function registerAll (Deprecated since 1.4.1, use SedonaContext.create() instead.).
  SedonaRegistrator.registerAll(sedona)


SedonaContext successfully created.

Running Test Query:
+-----------+
|pt         |
+-----------+
|POINT (0 0)|
+-----------+



In [15]:
# --- Zipcodes (ZCTA polygons) ⟷ WXAreas (polygons) using ST_Intersects ---
# Assumes `sedona` (SedonaContext SparkSession) already exists.

import time
from pyspark.sql import functions as F
from sedona.spark.register.geo_registrator import SedonaRegistrator
import os, glob, shutil


# ----------------------- CONFIG -----------------------
ZIPCODES_PATH = "/workspace/data/pa_subsets/pa_zcta_tl_2020_us_zcta520.parquet"
WXAREAS_PATH  = "/workspace/data/pa_subsets/pa_z_18mr25.parquet"
FINAL_CSV = "/workspace/data/sedona_geojoin_results.csv"
GEOM          = "geometry"
GRIDTYPE      = "kdbtree"   # or "quadtree"
BROADCAST_ZIP = True        # put zipcodes on broadcast side if it's smaller
# ------------------------------------------------------

# Ensure Sedona SQL functions are registered
try:
    SedonaRegistrator.registerAll(sedona)
except Exception:
    pass

sedona.sql(f"SET sedona.join.gridtype={GRIDTYPE}")

# ----------------------- LOAD -------------------------
zipcodes = sedona.read.format("geoparquet").load(ZIPCODES_PATH)
wxareas  = sedona.read.format("geoparquet").load(WXAREAS_PATH)

def ensure_geom(df):
    if GEOM in df.columns:
        return df
    for c in ("geom","the_geom","GEOMETRY","Geom"):
        if c in df.columns:
            return df.withColumnRenamed(c, GEOM)
    if "wkb" in df.columns:
        return df.withColumn(GEOM, F.expr("ST_GeomFromWKB(wkb)"))
    if "wkt" in df.columns:
        return df.withColumn(GEOM, F.expr("ST_GeomFromWKT(wkt)"))
    raise RuntimeError(f"No geometry-like column found in columns: {df.columns}")

zipcodes = ensure_geom(zipcodes)
wxareas  = ensure_geom(wxareas)

# -------------------- CRS NORMALIZATION ----------------
# Normalize BOTH to EPSG:4326 (safe for intersects)
zipcodes_4326 = zipcodes.withColumn(GEOM, F.expr("ST_Transform(geometry, 'EPSG:4269','EPSG:4326')"))
wxareas_4326  = wxareas.withColumn(GEOM,  F.expr("ST_Transform(geometry, 'EPSG:4269','EPSG:4326')"))

# If transform yields nulls (unlikely), fall back to as-is
if zipcodes_4326.filter(F.col(GEOM).isNotNull()).count() == 0:
    zipcodes_4326 = zipcodes
if wxareas_4326.filter(F.col(GEOM).isNotNull()).count() == 0:
    wxareas_4326 = wxareas

# -------------------- JOIN SQL (with correct hint placement) ----------------
def build_sql():
    hint = "/*+ BROADCAST(z) */ " if BROADCAST_ZIP else ""
    return f"""
      SELECT {hint} z.*, w.*
      FROM wxareas w
      JOIN zipcodes z
        ON ST_Intersects(z.{GEOM}, w.{GEOM})
    """

def try_join(z_df, w_df, label):
    z_df.createOrReplaceTempView("zipcodes")
    w_df.createOrReplaceTempView("wxareas")
    _ = sedona.sql("SELECT ST_AsText(ST_Point(0,0))").count()  # warm up functions/jars
    sql = build_sql()
    t0 = time.perf_counter()
    df = sedona.sql(sql)
    cnt = df.count()
    t1 = time.perf_counter()
    print(f"{label}: {cnt} rows in {t1 - t0:.2f}s")
    return cnt, df

# -------------------- COLD ----------------
c_cold, df_cold = try_join(zipcodes_4326, wxareas_4326, "❄️  COLD ST_Intersects")

# -------------------- WARM (cache inputs) ----------------
z_cached = zipcodes_4326.cache(); _ = z_cached.count()
w_cached = wxareas_4326.cache();  _ = w_cached.count()
c_warm, df_warm = try_join(z_cached, w_cached, "🔥  WARM ST_Intersects")

best_cnt, best_df = (c_warm if c_warm >= c_cold else c_cold), (df_warm if c_warm >= c_cold else df_cold)

# -------------------- WRITE CSV ----------------
if best_cnt > 0:
    best_df_selected = (
    best_df
    .withColumnRenamed("ZCTA5CE20", "zipcode")    # from zip polygons
    .withColumnRenamed("GEOID20", "GEOID20")      # keep as-is
    .withColumnRenamed("NAME", "weather_name")    # from wxareas
    .withColumnRenamed("CWA", "cwa")              # from wxareas
    .withColumnRenamed("ALAND20", "area_m2")      # land area (m²)
    .select("zipcode", "GEOID20", "weather_name", "cwa", "area_m2")
    )
    best_df_selected.toPandas().to_csv(FINAL_CSV,
                          index=False)
    print("✅ Wrote sedona_geojoin_results.csv (via pandas)")

    print(f"Δ warm/cold speed-up: {(c_warm / max(c_cold, 1)):.2f}×")
else:
    print("⚠️  0 rows intersected. If wxareas are actually points in this file, use ST_Covers instead of ST_Intersects.")



/tmp/ipykernel_2182/4103073422.py:21: DeprecationWarning: Call to deprecated function registerAll (Deprecated since 1.4.1, use SedonaContext.create() instead.).
  SedonaRegistrator.registerAll(sedona)
25/11/13 06:45:57 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/11/13 06:45:57 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/11/13 06:45:57 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/11/13 06:45:57 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/11/13 06:45:57 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/11/13 06:45:57 WARN SimpleFunctionRegistry: The function st_envelope_aggr replaced a previously registered function.
25/11/13 06:45:57 WARN

❄️  COLD ST_Intersects: 2928 rows in 25.39s


25/11/13 06:46:29 WARN CacheManager: Asked to cache already cached data.        


🔥  WARM ST_Intersects: 2928 rows in 6.06s


✅ Wrote sedona_geojoin_results.csv (via pandas)
Δ warm/cold speed-up: 1.00×


25/11/13 06:50:14 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...
25/11/13 06:50:14 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
25/11/13 06:50:14 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...
25/11/13 06:50:19 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.4: Remote RPC client disassociated. Likely due to containers exceeding thresholds, or network issues. Check driver logs for WARN messages.
25/11/13 06:50:20 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_530_6!
25/11/13 06:50:20 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_60_5!
25/11/13 06:50:20 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_60_7!
25/11/13 06:50:20 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_530_3!
25/11/13 06:50:20 WARN BlockManagerMaste